In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Load model
model_name = "microsoft/DialoGPT-medium"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

print("Chatbot: Hello! I am your AI assistant. Type 'exit' to stop.")

chat_history_ids = None

while True:
    user_input = input("You: ").strip()

    if user_input == "":
        continue

    print("You:", user_input)


    if user_input.lower() in ['exit', 'quit']:
        print("Chatbot: Goodbye! 👋")
        break

    # Encode input
    new_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt')

    # Append chat history
    if chat_history_ids is not None:
        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
    else:
        bot_input_ids = new_input_ids

    # Generate response
    chat_history_ids = model.generate(
        bot_input_ids,
        max_length=1000,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        top_k=40,
        top_p=0.9,
        temperature=0.7
    )

    # Keep only last 1000 tokens
    chat_history_ids = chat_history_ids[:, -1000:]

    # Decode response
    bot_response = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )

    print("Chatbot:", bot_response)


Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

Chatbot: Hello! I am your AI assistant. Type 'exit' to stop.
You: Hello
Chatbot: Hi, what's up?
You: exit
Chatbot: Goodbye! 👋
